Scarping for https://www.allfreecrochet.com/

In [27]:
import requests
import re
import time
import pandas as pd
from bs4 import BeautifulSoup
import os

headers = {"User-Agent": "Mozilla/5.0"}

image_dir = "../data/images"
os.makedirs(image_dir, exist_ok=True)

In [32]:
def scrape_product(product_link):
    row = {}
    os.makedirs(image_dir, exist_ok=True)

    try:
        r = requests.get(product_link, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(f"Connection error for {product_link}: {e}")
        return row

    # title
    title = soup.find("h1", class_="articleHeadline")
    if title:
        row["title"] = title.get_text(strip=True)
    else:
        row["title"] = ""  

    # description
    desc = soup.select_one("div.articleAttrSection")
    description = desc.get_text(strip=True)
    row["description"] = description

    sections = soup.select("div.articleAttrSection")

    # find details in the sections
    for section in sections:
        label_span = section.find("span", class_="attrLabel")
        img = section.find("img")

        if label_span:
            key = label_span.get_text(strip=True)
            
            # check if next span exist only in this section
            value_span = label_span.find_next_siblings("span")

            if value_span:
                value = value_span[0].get_text(strip=True)
            else:
                # No span sibling, get text in the parent <p> minus the label
                full_text = label_span.parent.get_text(" ", strip=True)
                # remove the label text from the beginning
                value = full_text.replace(key, "", 1).strip()
            
            row[key] = value

        # skill level is in img tag, so check it separately
        elif img:
            row["skill_level"] = img.get("alt", "").strip()

    # image extraction and download
    img_container = soup.find("div", class_="imgContainer")
    img_tag = img_container.find("img") if img_container else None

    img_url = None
    if img_tag:
        img_url = "https:" + img_tag.get("src")
    row["img_url"] = img_url

    if img_url:
        safe_filename = re.sub(r'[^\w\s-]', '', row["title"]).strip().replace(' ', '_')
        save_path = os.path.join(image_dir, f"{safe_filename}.jpg")

        try:
            img_res = requests.get(img_url, headers=headers, stream=True, timeout=10)
            if img_res.status_code == 200 and 'image' in img_res.headers.get('Content-Type', ''):
                with open(save_path, 'wb') as f:
                    for chunk in img_res.iter_content(1024):
                        f.write(chunk)
                row["local_path"] = save_path
            else:
                print(f"Invalid image response for: {row['title']}")
        except Exception as e:
            print(f"Error downloading image for {row['title']}: {e}")

    return row


In [33]:
test = "https://www.allfreecrochet.com/Crochet-Afghan-Patterns/Spring-Breeze-Baby-Blanket-187303"
scrape_product(test)

{'title': 'Spring Breeze Baby Blanket',
 'description': '"Wrap your little one in the soft, airy charm of the Spring Breeze Baby Blanket – Free Crochet Pattern that’s as refreshing as a warm spring morning! This delightful design features a soothing rhythm of two rows of cozy double crochet, followed by a playful eyelet row that adds just the right touch of texture. Then, just when you think you’ve got the pattern down, a second, slightly different eyelet row keeps things interesting and fun for your hook. The result is a light and breezy baby blanket with a beautiful balance of simplicity and whimsy which is perfect for springtime snuggles or thoughtful handmade gifts!"',
 'skill_level': 'Intermediate',
 'Crochet Hook': 'H/8 or 5 mm hook',
 'Yarn Weight': '(4) Medium Weight/Worsted Weight and Aran (16-20 stitches to 4 inches)',
 'Finished Size': '26.45 x 25.60 inches approx',
 'img_url': 'https://irepo.primecp.com/2025/06/601674/1749590604_117446_Large600_ID-5853120.png?v=5853120',
 '

In [ ]:
base_url = "https://www.allfreecrochet.com"
rows = []

response = requests.get(base_url, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(response.text, "html.parser")

cards = soup.select("div.articleCell")

for card in cards:
    link = card.select_one("a[href]")
    product_link = base_url + link["href"] 

    img_tag = card.select_one("img")
    image_link = img_tag["src"] if img_tag else None

    name_tag = card.select_one("figcaption a")
    name = name_tag.get_text(strip=True) if name_tag else None

    product_data = scrape_product(product_link)

    product_data["name"] = name
    product_data["pattern_link"] = product_link
    product_data["image_link"] = image_link

    rows.append(product_data)
    time.sleep(1)


print("Scraping completed. Total products scraped:", len(rows))
df = pd.DataFrame(rows)

In [24]:
df.head()

,description,Notes,skill_level,Crochet Hook,Yarn Weight,Crochet Gauge,Finished Size,name,pattern_link,image_link
0,Test your crochet skills with this amazing Rai...,1. Work all rows with RS of work facing. 2. To...,Intermediate,M/13 or 9 mm hook,(5) Bulky/Chunky (12-15 stitches for 4 inches),"9 sts and 10 rows = 4"" [10 cm].","approx 35"" x 42"" [89 x 106.5 cm].",Rainbow Tunisian Entrelac Crochet Baby Blanket...,https://www.allfreecrochet.com/Baby-Afghan-Cro...,//irepo.primecp.com/2015/07/227100/bernat-soft...
1,We’ve all been there: you’re digging through y...,NaN,NaN,NaN,NaN,NaN,NaN,47+ One Skein Crochet Patterns,https://www.allfreecrochet.com/Miscellaneous-C...,//irepo.primecp.com/2019/04/407909/AFC-One-Ske...
2,If you've been searching for easy afghan patte...,NaN,Beginner,H/8 or 5 mm hook,(4) Medium Weight/Worsted Weight and Aran (16-...,"13 sc and 14 rows = 4"" (10cm)",Measurement: 48 x 60”,World's Easiest Afghan,https://www.allfreecrochet.com/Crochet-Afghan-...,//irepo.primecp.com/2014/12/204013/Worlds-Easi...
3,"""We absolutely love long scarves. The type tha...",NaN,Easy,H/8 or 5 mm hook,(4) Medium Weight/Worsted Weight and Aran (16-...,15 stitches and 8 rows = 10 cm in double crochet,Made to Measure,Spring Oversized Long Crochet Scarf,https://www.allfreecrochet.com/Scarves/Spring-...,//irepo.primecp.com/2025/07/602825/1752522717_...
4,"""Wrap your little one in the soft, airy charm ...",NaN,Intermediate,H/8 or 5 mm hook,(4) Medium Weight/Worsted Weight and Aran (16-...,NaN,26.45 x 25.60 inches approx,Spring Breeze Baby Blanket,https://www.allfreecrochet.com/Crochet-Afghan-...,//irepo.primecp.com/2025/06/601674/1749590604_...
